In [1]:
# RAG의 임베딩·검색·생성 단계에 필요한 패키지를 설치합니다.
# -q는 설치 로그를 줄이고, ==는 예제가 검증된 버전을 사용하게 합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>RAG의 생성 단계는 검색한 문서와 질문을 함께 모델에 전달합니다. 출처 태그를 문맥에 포함하고 답에도 같은 형식으로 표시하도록 요청합니다.</p>

In [2]:
# find_dotenv(usecwd=True)는 노트북 작업 폴더를 기준으로 .env 경로를 반환합니다.
# load_dotenv()는 그 파일을 읽고, os.getenv()는 OPENAI_API_KEY 값이 로드되었는지 확인합니다.
# 두 assert는 준비가 되지 않은 상태에서 외부 API를 호출하지 않도록 막습니다.
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>답과 출처가 분명한 문서 두 건을 실제 임베딩으로 색인합니다.</p>

In [3]:
# Document의 page_content는 검색할 본문이고 metadata는 doc_id와 page 출처 정보입니다.
# from_documents()는 docs를 임베딩해 InMemoryVectorStore를 만들어 store에 반환합니다.
# text-embedding-3-small은 문서와 이후 질문을 같은 벡터 공간에 표현할 모델입니다.
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

docs = [Document(page_content='정기 휴관일은 매주 월요일입니다.',
                 metadata={'doc_id': 'D01', 'page': 1}),
        Document(page_content='대출 기간은 14일입니다.',
                 metadata={'doc_id': 'D02', 'page': 2})]
store = InMemoryVectorStore.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'))


<h3>검색 결과로 답 생성</h3>
<p>검색한 문서의 ID와 쪽 번호를 문맥에 넣고 실제 생성 모델에 전달합니다.</p>

In [4]:
# hits는 question과 가장 가까운 Document 한 건을 담은 리스트입니다.
# join()과 제너레이터 표현식은 각 문서의 출처·본문을 줄바꿈으로 연결해 context를 만듭니다.
# f 삼중 문자열이 context와 question을 프롬프트에 넣고, invoke()는 AIMessage를 반환합니다.
question = '정기 휴관일은 언제인가요?'
hits = store.similarity_search(question, k=1)
context = '\n'.join(
    f"[{d.metadata['doc_id']} p.{d.metadata['page']}] {d.page_content}" for d in hits
)
prompt = f"""문서만 근거로 답하고 출처를 붙이세요.
[문서]
{context}
[질문] {question}"""
answer = init_chat_model('openai:gpt-5.4-mini', temperature=0).invoke(prompt)
print(answer.text)


정기 휴관일은 **매주 월요일**입니다. [D01 p.1]


<h3>변형 실행</h3>
<p><code>k</code>를 2로 바꿨을 때 모델에게 전달되는 문맥이 어떻게 늘어나는지 확인합니다.</p>

In [5]:
# k=2로 같은 question을 다시 검색해 문맥이 두 건으로 늘어난 결과를 more_hits에 받습니다.
# 리스트 표기식은 각 Document에서 doc_id만 추출해 검색 순서를 출력합니다.
more_hits = store.similarity_search(question, k=2)
print([d.metadata['doc_id'] for d in more_hits])


['D01', 'D02']


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [6]:
# 첫 assert는 생성 응답에 휴관일 답이 있는지 검사합니다.
# 둘째 assert의 and는 문서 ID와 쪽 번호가 둘 다 표시되어야 통과하게 합니다.
assert '월요일' in answer.text
assert 'D01' in answer.text and 'p.1' in answer.text
print('검증 통과: 실제 생성 응답에 답과 출처가 있습니다.')


검증 통과: 실제 생성 응답에 답과 출처가 있습니다.


<h3>정리</h3>
<p>출처 표기는 생성 단계에서 갑자기 생기지 않습니다. 로더가 남긴 메타데이터를 검색 결과와 함께 프롬프트에 전달해야 답의 근거를 확인할 수 있습니다.</p>

<p><strong>원인</strong> — <code>page</code> 메타데이터가 없으면 출처 태그를 만드는 코드에서 <code>KeyError</code>가 발생합니다.</p>

In [7]:
# broken은 doc_id만 있고 page가 없는 의도적으로 불완전한 Document입니다.
# f-string에서 metadata['page']를 읽으면 KeyError가 발생하고 except가 누락된 키를 error에 담습니다.
# 예외를 포착해 실패를 출력하므로 뒤의 After 셀까지 계속 실행할 수 있습니다.
broken = Document(page_content='휴관일은 월요일입니다.',
                  metadata={'doc_id': 'D01'})
try:
    print(f"[{broken.metadata['doc_id']} p.{broken.metadata['page']}]")
except KeyError as error:
    print('예상한 실패: 누락된 메타데이터', error)


예상한 실패: 누락된 메타데이터 'page'


<p><strong>수정</strong> — 문서를 만들 때 쪽 번호를 메타데이터에 명시하고 출처 태그를 다시 만듭니다.</p>

In [8]:
# fixed는 broken의 본문을 재사용하되 page=1을 명시한 새 Document입니다.
# f-string은 doc_id와 page를 [D01 p.1] 형식으로 조합해 fixed_tag를 반환합니다.
# assert는 수정한 메타데이터로 예상한 출처 태그가 정확히 만들어졌는지 확인합니다.
fixed = Document(page_content=broken.page_content,
                 metadata={'doc_id': 'D01', 'page': 1})
fixed_tag = f"[{fixed.metadata['doc_id']} p.{fixed.metadata['page']}]"
print('After:', fixed_tag)
assert fixed_tag == '[D01 p.1]'


After: [D01 p.1]
